In [1]:
from pathlib import Path
import urllib.request
import zipfile
import numpy as np
import pandas as pd

# Student-Specific Configuration
LAST_NAME = "Pascual"
STUDENT_ID = "TUPM-26-4321"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])          # 1
ID_SUM = sum(map(int, id_digits))        # 18
NAME_LENGTH = len(LAST_NAME)             # 7
RANDOM_SEED = int(id_digits) % (2**32)   # 264321

# Output directory setup
WORK_DIR = Path.cwd()
if (WORK_DIR / "Lab7_NumPy_Pandas_Additional_Exercises").exists():
    OUT_DIR = WORK_DIR / "Lab7_NumPy_Pandas_Additional_Exercises"
else:
    OUT_DIR = WORK_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

# Download & Extract UCI SECOM Dataset if not already present
zip_path = DATA_DIR / "secom.zip"
data_file = DATA_DIR / "secom.data"
labels_file = DATA_DIR / "secom_labels.data"

if not (data_file.exists() and labels_file.exists()):
    url = "https://archive.ics.uci.edu/static/public/179/secom.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)

# Load SECOM measurement matrix and labels preserving original order
X_raw = np.loadtxt(data_file)
labels_df = pd.read_csv(labels_file, sep=r"\s+", header=None, names=["Result", "Timestamp"])
y_raw = labels_df["Result"].to_numpy()
timestamps = labels_df["Timestamp"].to_numpy()

n_samples, n_features = X_raw.shape
feature_names = np.array([f"feature_{i:03d}" for i in range(n_features)])

print(f"Student: {LAST_NAME} ({STUDENT_ID})")
print(f"SEED_DIGIT={SEED_DIGIT}, ID_SUM={ID_SUM}, NAME_LENGTH={NAME_LENGTH}, RANDOM_SEED={RANDOM_SEED}")
print(f"SECOM Matrix Shape: {X_raw.shape} | Pass (-1): {(y_raw == -1).sum()}, Fail (1): {(y_raw == 1).sum()}")

Student: Pascual (TUPM-26-4321)
SEED_DIGIT=1, ID_SUM=18, NAME_LENGTH=7, RANDOM_SEED=264321
SECOM Matrix Shape: (1567, 590) | Pass (-1): 1463, Fail (1): 104


In [2]:
# 1. Identify features containing all-missing or constant observed values
all_missing_mask = np.isnan(X_raw).all(axis=0)
obs_min = np.nanmin(np.where(all_missing_mask, 0.0, X_raw), axis=0)
obs_max = np.nanmax(np.where(all_missing_mask, 0.0, X_raw), axis=0)
constant_mask = (~all_missing_mask) & np.isclose(obs_min, obs_max, equal_nan=True)

usable_indices = np.where(~(all_missing_mask | constant_mask))[0]

# 2. Reproducibly select 5 + (SEED_DIGIT % 3) usable features
rng1 = np.random.default_rng(RANDOM_SEED)
n_select_ex1 = 5 + (SEED_DIGIT % 3)  # 5 + 1 = 6 features
selected_idx_ex1 = np.sort(rng1.choice(usable_indices, size=n_select_ex1, replace=False))
selected_feats_ex1 = feature_names[selected_idx_ex1]

X_sub1 = X_raw[:, selected_idx_ex1]

# 3. Feature-wise means, medians, sample SDs, and missing percentages using NumPy
means_1 = np.nanmean(X_sub1, axis=0)
medians_1 = np.nanmedian(X_sub1, axis=0)
stds_1 = np.nanstd(X_sub1, axis=0, ddof=1)
missing_pct_1 = np.isnan(X_sub1).mean(axis=0) * 100.0

# 4. Robust standardized values using median and MAD (safe handling for MAD == 0)
mad_1 = np.nanmedian(np.abs(X_sub1 - medians_1), axis=0)
scaled_mad_1 = 1.4826 * mad_1
safe_denom_1 = np.where(scaled_mad_1 == 0, np.nan, scaled_mad_1)
robust_z_1 = (X_sub1 - medians_1) / safe_denom_1

# 5. Identify measurements with |robust_z| > 3.5 (Boolean indexing)
flagged_mask_1 = np.abs(robust_z_1) > 3.5
flagged_counts_1 = np.sum(flagged_mask_1, axis=0)
observed_counts_1 = np.sum(~np.isnan(X_sub1), axis=0)
flagged_pct_1 = (flagged_counts_1 / observed_counts_1) * 100.0

# Verify vectorized calculation against element-wise loop
loop_flags = sum(
    1
    for i in range(X_sub1.shape[0])
    for j in range(X_sub1.shape[1])
    if not np.isnan(robust_z_1[i, j]) and abs(robust_z_1[i, j]) > 3.5
)
vectorized_verified = bool(int(np.sum(flagged_mask_1)) == loop_flags)

# 6. Rank features by percentage of flagged observations and export CSV
order_ex1 = np.argsort(-flagged_pct_1)
mq_df = pd.DataFrame({
    "Rank": np.arange(1, n_select_ex1 + 1),
    "Feature": selected_feats_ex1[order_ex1],
    "Mean": np.round(means_1[order_ex1], 4),
    "Median": np.round(medians_1[order_ex1], 4),
    "Sample_SD": np.round(stds_1[order_ex1], 4),
    "MAD": np.round(mad_1[order_ex1], 4),
    "Missing_Pct": np.round(missing_pct_1[order_ex1], 4),
    "Flagged_Count": flagged_counts_1[order_ex1],
    "Flagged_Pct": np.round(flagged_pct_1[order_ex1], 4),
})

mq_path = OUT_DIR / "measurement_quality.csv"
mq_df.to_csv(mq_path, index=False)

print("=== EXERCISE 1 ASSESSMENT DATA ===")
print("Selected Features:", ", ".join(selected_feats_ex1))
print(f"Highest Missing Percentage: {missing_pct_1.max():.4f}% ({selected_feats_ex1[np.argmax(missing_pct_1)]})")
print(f"Feature with Highest Flagged Percentage: {mq_df.iloc[0]['Feature']} ({mq_df.iloc[0]['Flagged_Pct']:.4f}%)")
print(f"Total Flagged Observations: {int(np.sum(flagged_mask_1))}")
print(f"Vectorized Results Verified: {vectorized_verified}")
print("\nRanked Summary Table:")
print(mq_df.to_string(index=False))

=== EXERCISE 1 ASSESSMENT DATA ===
Selected Features: feature_168, feature_215, feature_297, feature_490, feature_500, feature_549
Highest Missing Percentage: 16.5922% (feature_549)
Feature with Highest Flagged Percentage: feature_297 (19.6166%)
Total Flagged Observations: 396
Vectorized Results Verified: True

Ranked Summary Table:
 Rank     Feature      Mean   Median  Sample_SD      MAD  Missing_Pct  Flagged_Count  Flagged_Pct
    1 feature_297 2342.8270 820.0988  3226.9243 754.8589       0.1276            307      19.6166
    2 feature_549    0.6633   0.4710     0.6733   0.2475      16.5922             61       4.6672
    3 feature_490   51.3540  48.5574    18.0486  11.0878       0.0638             14       0.8940
    4 feature_168    0.1244   0.1190     0.0476   0.0300       0.1276              8       0.5112
    5 feature_215    0.0811   0.0825     0.0304   0.0170       1.5316              6       0.3889
    6 feature_500  240.9814   0.0000   323.0034   0.0000       0.1276        

In [3]:
# 1. Construct labeled DataFrame preserving original SECOM record identifiers
df_secom = pd.DataFrame(X_raw, columns=feature_names)
df_secom.insert(0, "Record_ID", np.arange(n_samples))
df_secom.insert(1, "Timestamp", timestamps)
df_secom.insert(2, "Result", y_raw)
df_secom.insert(3, "Outcome", np.where(y_raw == 1, "Fail", "Pass"))

# 2. Reproducible balanced sample containing min(20 + ID_SUM, Pass count, Fail count) per outcome
target_per_group = min(20 + ID_SUM, int((y_raw == -1).sum()), int((y_raw == 1).sum()))  # min(38, 1463, 104) = 38
rng2 = np.random.default_rng(RANDOM_SEED + 2)

pass_ids = df_secom.loc[df_secom["Outcome"] == "Pass", "Record_ID"].to_numpy()
fail_ids = df_secom.loc[df_secom["Outcome"] == "Fail", "Record_ID"].to_numpy()

sampled_pass_ids = np.sort(rng2.choice(pass_ids, size=target_per_group, replace=False))
sampled_fail_ids = np.sort(rng2.choice(fail_ids, size=target_per_group, replace=False))
balanced_ids = np.concatenate([sampled_pass_ids, sampled_fail_ids])

balanced_df = df_secom.loc[balanced_ids].copy().reset_index(drop=True)

# 3. Select 5 usable features and apply median imputation using medians from the FULL dataset
selected_idx_ex2 = np.sort(rng2.choice(usable_indices, size=5, replace=False))
selected_feats_ex2 = list(feature_names[selected_idx_ex2])

full_medians_ex2 = df_secom[selected_feats_ex2].median()
balanced_imputed = balanced_df[["Record_ID", "Timestamp", "Result", "Outcome"] + selected_feats_ex2].copy()
balanced_imputed[selected_feats_ex2] = balanced_imputed[selected_feats_ex2].fillna(full_medians_ex2)

balanced_sample_path = OUT_DIR / "balanced_sample.csv"
balanced_imputed.to_csv(balanced_sample_path, index=False)

# 4. Use .groupby() and .agg() to calculate counts, means, medians, and sample SDs
grouped_ex2 = balanced_imputed.groupby("Outcome")[selected_feats_ex2].agg(["count", "mean", "median", "std"])

# 5. Absolute mean difference and standardized mean difference (pooled within-group sample SD)
comparison_rows = []
np_pd_checks = []

for feat in selected_feats_ex2:
    n_pass = grouped_ex2.loc["Pass", (feat, "count")]
    n_fail = grouped_ex2.loc["Fail", (feat, "count")]
    mean_pass = grouped_ex2.loc["Pass", (feat, "mean")]
    mean_fail = grouped_ex2.loc["Fail", (feat, "mean")]
    med_pass = grouped_ex2.loc["Pass", (feat, "median")]
    med_fail = grouped_ex2.loc["Fail", (feat, "median")]
    sd_pass = grouped_ex2.loc["Pass", (feat, "std")]
    sd_fail = grouped_ex2.loc["Fail", (feat, "std")]

    # Cross-check with NumPy directly
    np_mean_fail = np.mean(balanced_imputed.loc[balanced_imputed["Outcome"] == "Fail", feat].to_numpy())
    np_sd_fail = np.std(balanced_imputed.loc[balanced_imputed["Outcome"] == "Fail", feat].to_numpy(), ddof=1)
    np_pd_checks.append(np.isclose(mean_fail, np_mean_fail) and np.isclose(sd_fail, np_sd_fail))

    abs_diff = abs(mean_fail - mean_pass)
    pooled_var = (((n_pass - 1) * (sd_pass ** 2)) + ((n_fail - 1) * (sd_fail ** 2))) / (n_pass + n_fail - 2)
    pooled_sd = np.sqrt(pooled_var)
    smd = 0.0 if pooled_sd == 0 else (mean_fail - mean_pass) / pooled_sd

    comparison_rows.append({
        "Feature": feat,
        "Pass_Count": int(n_pass),
        "Fail_Count": int(n_fail),
        "Pass_Mean": round(mean_pass, 4),
        "Fail_Mean": round(mean_fail, 4),
        "Pass_Median": round(med_pass, 4),
        "Fail_Median": round(med_fail, 4),
        "Pass_SD": round(sd_pass, 4),
        "Fail_SD": round(sd_fail, 4),
        "Abs_Mean_Diff": round(abs_diff, 4),
        "Pooled_SD": round(pooled_sd, 4),
        "Std_Mean_Diff": round(smd, 4),
        "Abs_Std_Mean_Diff": round(abs(smd), 4),
    })

# 6. Rank by absolute standardized mean difference and export
comp_df = pd.DataFrame(comparison_rows).sort_values(by="Abs_Std_Mean_Diff", ascending=False).reset_index(drop=True)
comp_df.insert(0, "Rank", np.arange(1, len(comp_df) + 1))

comp_path = OUT_DIR / "feature_comparison.csv"
comp_df.to_csv(comp_path, index=False)

print("=== EXERCISE 2 ASSESSMENT DATA ===")
print(f"Balanced Sample Size: {len(balanced_imputed)} ({target_per_group} Pass, {target_per_group} Fail)")
print("Selected Features:", ", ".join(selected_feats_ex2))
print(f"Highest-Ranked Feature: {comp_df.iloc[0]['Feature']}")
print(f"Absolute Standardized Mean Difference: {comp_df.iloc[0]['Abs_Std_Mean_Diff']:.4f}")
print(f"NumPy-Pandas Statistics Match: {all(np_pd_checks)}")
print("\nFeature Comparison Table:")
print(comp_df[["Rank", "Feature", "Pass_Mean", "Fail_Mean", "Abs_Mean_Diff", "Abs_Std_Mean_Diff"]].to_string(index=False))

=== EXERCISE 2 ASSESSMENT DATA ===
Balanced Sample Size: 76 (38 Pass, 38 Fail)
Selected Features: feature_072, feature_095, feature_103, feature_345, feature_499
Highest-Ranked Feature: feature_103
Absolute Standardized Mean Difference: 0.8493
NumPy-Pandas Statistics Match: True

Feature Comparison Table:
 Rank     Feature  Pass_Mean  Fail_Mean  Abs_Mean_Diff  Abs_Std_Mean_Diff
    1 feature_103    -0.0094    -0.0070         0.0024             0.8493
    2 feature_095     0.0000     0.0001         0.0001             0.6496
    3 feature_072   152.1586   150.2503         1.9083             0.2253
    4 feature_499   256.1738   212.9372        43.2366             0.1320
    5 feature_345     8.0918     8.3942         0.3024             0.0202


In [4]:
# 1. Reproducibly select 5 usable features with at least 10 observed measurements in each outcome group
pass_obs_counts = np.sum(~np.isnan(X_raw[y_raw == -1]), axis=0)
fail_obs_counts = np.sum(~np.isnan(X_raw[y_raw == 1]), axis=0)
eligible_idx_ex3 = np.where((~(all_missing_mask | constant_mask)) & (pass_obs_counts >= 10) & (fail_obs_counts >= 10))[0]

rng3 = np.random.default_rng(RANDOM_SEED + 3)
selected_idx_ex3 = np.sort(rng3.choice(eligible_idx_ex3, size=5, replace=False))
selected_feats_ex3 = list(feature_names[selected_idx_ex3])

# 2 & 3. Two preparation strategies: (A) Available-Case Analysis vs (B) Full-Dataset Median Imputation
B_REPS = 200 + 10 * SEED_DIGIT  # 200 + 10 = 210 repetitions

def run_bootstrap_pipeline(seed_val):
    rng_boot = np.random.default_rng(seed_val)
    results = []

    pass_pool = np.where(y_raw == -1)[0]
    fail_pool = np.where(y_raw == 1)[0]
    boot_pass_idx = rng_boot.choice(pass_pool, size=(B_REPS, len(pass_pool)), replace=True)
    boot_fail_idx = rng_boot.choice(fail_pool, size=(B_REPS, len(fail_pool)), replace=True)

    for col_i, feat in zip(selected_idx_ex3, selected_feats_ex3):
        col_vals = X_raw[:, col_i]
        full_med = np.nanmedian(col_vals)
        col_imp = np.where(np.isnan(col_vals), full_med, col_vals)

        avail_diff = np.nanmean(col_vals[fail_pool]) - np.nanmean(col_vals[pass_pool])
        imp_diff = np.mean(col_imp[fail_pool]) - np.mean(col_imp[pass_pool])

        avail_boot_diffs = np.nanmean(col_vals[boot_fail_idx], axis=1) - np.nanmean(col_vals[boot_pass_idx], axis=1)
        imp_boot_diffs = np.mean(col_imp[boot_fail_idx], axis=1) - np.mean(col_imp[boot_pass_idx], axis=1)

        avail_boot_mean = float(np.nanmean(avail_boot_diffs))
        avail_ci_low, avail_ci_high = np.nanpercentile(avail_boot_diffs, [2.5, 97.5])

        imp_boot_mean = float(np.mean(imp_boot_diffs))
        imp_ci_low, imp_ci_high = np.percentile(imp_boot_diffs, [2.5, 97.5])

        results.append({
            "Feature": feat,
            "Avail_Obs_Diff": avail_diff,
            "Avail_Boot_Mean_Diff": avail_boot_mean,
            "Avail_CI_Low": float(avail_ci_low),
            "Avail_CI_High": float(avail_ci_high),
            "Imp_Obs_Diff": imp_diff,
            "Imp_Boot_Mean_Diff": imp_boot_mean,
            "Imp_CI_Low": float(imp_ci_low),
            "Imp_CI_High": float(imp_ci_high),
            "Diff_Change": abs(imp_boot_mean - avail_boot_mean),
            "Sign_Changed": bool(np.sign(avail_boot_mean) != np.sign(imp_boot_mean)),
        })

    df_res = pd.DataFrame(results)
    df_res["Avail_Rank"] = df_res["Avail_Boot_Mean_Diff"].abs().rank(ascending=False).astype(int)
    df_res["Imp_Rank"] = df_res["Imp_Boot_Mean_Diff"].abs().rank(ascending=False).astype(int)
    df_res["Rank_Changed"] = df_res["Avail_Rank"] != df_res["Imp_Rank"]
    return df_res

rob_df = run_bootstrap_pipeline(RANDOM_SEED + 30)
rob_df_verify = run_bootstrap_pipeline(RANDOM_SEED + 30)
repro_verified = bool(np.allclose(rob_df["Avail_Boot_Mean_Diff"], rob_df_verify["Avail_Boot_Mean_Diff"]) and
                      np.allclose(rob_df["Imp_Boot_Mean_Diff"], rob_df_verify["Imp_Boot_Mean_Diff"]))

rob_export = rob_df.copy()
num_cols = ["Avail_Obs_Diff", "Avail_Boot_Mean_Diff", "Avail_CI_Low", "Avail_CI_High",
            "Imp_Obs_Diff", "Imp_Boot_Mean_Diff", "Imp_CI_Low", "Imp_CI_High", "Diff_Change"]
rob_export[num_cols] = rob_export[num_cols].round(4)

rob_path = OUT_DIR / "robustness_comparison.csv"
rob_export.to_csv(rob_path, index=False)

largest_change_row = rob_export.loc[rob_export["Diff_Change"].idxmax()]
sign_changed_feats = rob_export.loc[rob_export["Sign_Changed"], "Feature"].tolist()
rank_changed_feats = rob_export.loc[rob_export["Rank_Changed"], "Feature"].tolist()

sign_str = ", ".join(sign_changed_feats) if sign_changed_feats else "None (0 features)"
rank_str = ", ".join(rank_changed_feats) if rank_changed_feats else "None (0 features)"

report_text = f"""STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT
Student: {LAST_NAME} ({STUDENT_ID})
Random Seed: {RANDOM_SEED} | Bootstrap Repetitions: {B_REPS}
Selected Features: {", ".join(selected_feats_ex3)}

1. Summary of Findings:
   - Largest Change in Bootstrap Mean Difference: {largest_change_row['Diff_Change']:.4f} ({largest_change_row['Feature']})
   - Features with Changed Difference Sign: {sign_str}
   - Features with Changed Ranking: {rank_str}
   - Reproducibility Verified: {repro_verified}

2. Engineering Interpretation:
   - Comparing available-case analysis against full-dataset median imputation shows how missing-value
     handling affects Fail-minus-Pass mean differences and relative feature importance.
   - Features whose 95% bootstrap confidence intervals exclude zero demonstrate consistent directional
     separation between failing and passing wafers across repeated stratified resampling.
"""

report_path = OUT_DIR / "sensitivity_report.txt"
report_path.write_text(report_text, encoding="utf-8")

print("=== EXERCISE 3 ASSESSMENT DATA ===")
print("Selected Features:", ", ".join(selected_feats_ex3))
print(f"Bootstrap Repetitions: {B_REPS}")
print(f"Largest Change in Mean Difference: {largest_change_row['Diff_Change']:.4f} ({largest_change_row['Feature']})")
print(f"Features with Changed Difference Sign: {sign_str}")
print(f"Features with Changed Ranking: {rank_str}")
print(f"Reproducibility Verified: {repro_verified}")
print("\nRobustness Comparison Table:")
print(rob_export[["Feature", "Avail_Boot_Mean_Diff", "Imp_Boot_Mean_Diff", "Diff_Change", "Avail_Rank", "Imp_Rank"]].to_string(index=False))

=== EXERCISE 3 ASSESSMENT DATA ===
Selected Features: feature_054, feature_105, feature_115, feature_224, feature_431
Bootstrap Repetitions: 210
Largest Change in Mean Difference: 0.2080 (feature_431)
Features with Changed Difference Sign: None (0 features)
Features with Changed Ranking: None (0 features)
Reproducibility Verified: True

Robustness Comparison Table:
    Feature  Avail_Boot_Mean_Diff  Imp_Boot_Mean_Diff  Diff_Change  Avail_Rank  Imp_Rank
feature_054                0.0034              0.0034       0.0000           4         4
feature_105               -0.0000             -0.0000       0.0000           5         5
feature_115               -8.3940             -8.3940       0.0000           2         2
feature_224                0.0038              0.0042       0.0004           3         3
feature_431               17.8448             17.6368       0.2080           1         1


In [5]:
!zip additional_outputs.zip measurement_quality.csv balanced_sample.csv feature_comparison.csv robustness_comparison.csv sensitivity_report.txt
from google.colab import files
files.download("additional_outputs.zip")

  adding: measurement_quality.csv (deflated 42%)
  adding: balanced_sample.csv (deflated 69%)
  adding: feature_comparison.csv (deflated 54%)
  adding: robustness_comparison.csv (deflated 58%)
  adding: sensitivity_report.txt (deflated 42%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>